In [7]:
import torch
import learnQorthogonal as lqOrth
import numpy as np
rng = np.random.default_rng(seed=40)
torch.manual_seed(212)

# number of time points and units
T = 7
N = 8

# number of factors
M = 2
K = 3

In [8]:
# --------------------- Factors ---------------------
def generate_factors():
    scale = 0.1
    # lists of normal rvs
    eps1 = rng.normal(0, scale, size=T)
    eps2 = rng.normal(0, scale, size=T)
    eps3 = rng.normal(0, scale, size=T)

    f1, f2, f3 = np.zeros(T), np.zeros(T), np.zeros(T)

    for t in range(T):
        # from Rong's suggestion
        f1[t] = eps1[t] + (0.8*f1[t-1] if t >= 1 else 0)
        f2[t] = eps2[t] + (-0.6*f2[t-1] + 0.8*eps2[t-1] if t >= 1 else 0)
        f3[t] = eps3[t] + (0.9*eps3[t-1] if t >= 1 else 0) + (0.4*eps3[t-2] if t >= 2 else 0)
    return f1, f2, f3

#-----------------------------------------------------

In [9]:
# --------------------- Loadings -------------------------------

# a function to create loadings
def generate_loadings(m):
    rho = 1
    Sigma = (1 - rho) * np.eye(m) + rho * np.ones((m, m))

    # list in which to collect the loadings
    loadings_all = []

    # mean vector for each factor - length of this vector is M, and we have K of them
    # mean vector is quite similar for each K 
    means = [rng.normal(0,0.5, size=m) for _ in range(K)]
    # for each unit, draw loadings for each factor
    for i in range(N):
        # the loadings for unit i by the end of the loop.
        loadings = [] # K lists of length M
        for k in range(K):
            # draw loadings from normal distribution with mean mean_k
            lambda_ik = rng.multivariate_normal(mean=means[k], cov=Sigma) 
            loadings.append(lambda_ik)
        # add the loadings for unit i to the collection of all loadings.
        loadings_all.append(loadings)
    
    return loadings_all
#--------------------------------------------------------------

In [ ]:
# --------------------- Outcomes plus Noise ---------------------

# function to multiply factors by loadings
def generate_noiseless_outcomes(loadings_all, f1,f2,f3):
    # extract the factors over time
    factors = [np.array([f1[t], f2[t], f3[t]]) for t in range(T)]
    # to store all outcomes
    outcomes = []

    # for each unit
    for i in range(N):
        # initialize list to store outcomes
        outcomes_i = []
        # calculate the outcomes at time t
        for t in range(T):
            # the outcome at time t is the factors at t times the loadings
            f_t = factors[t]
            # K x M (3 x 2). So Lambda_i.T = (2,3). f1[t],f2[t],f3[t] is (3,)?
            Lambda_i = np.array(loadings_all[i]) 
            Y_it = np.dot(Lambda_i.T, f_t) 
            # print(f"{i},{t}, {Y_it}")
            outcomes_i.append(Y_it)
        # add the outcomes for unit i to the collection of all outcomes
        outcomes.append(outcomes_i)
    # transform to np array
    outcomes=np.array(outcomes)

    # ADD NOISE HERE - Use an array of dimension outcomes.shape and each entry is std normal.

    # turn the outcomes in np arrays
    for i in range(len(outcomes)):
        outcomes[i] = torch.tensor(outcomes[i]) # shape (T, M)

    # print("all outcomes:", outcomes)
    return outcomes

#-------------------------------------------------------------------------

In [11]:
#------------------ Shape the outcomes to be fed to the model ------------
def reshape_data(outcomes):
    # split outcomes into treated outcomes and control outcomes
    treated = torch.tensor(outcomes[0]) # (1, T, M)
    control = torch.tensor(outcomes[1:]) # (N-1, T, M)

    # break outcomes down into observations at each time point:
    treated_outcomes_list = []
    for t in range(T):
        treated_t = treated[t, :].unsqueeze(1) # (M, 1), <class 'torch.Tensor'>
        treated_outcomes_list.append(treated_t)

    # print("treated:", treated_outcomes_list)

    # break control outcomes down into observations at each time point:
    control_outcomes_list = []
    for t in range(T):
        # grab the all outcomes from all control units at time t
        control_t = control[:, t, :].T # (N-1, M), <class 'torch.Tensor'>
        control_outcomes_list.append(control_t)

    return treated_outcomes_list, control_outcomes_list

    # print("control:", control_outcomes_list)

In [12]:
#--------------- Calculate correlation across outcomes ---------
# TURN THIS INTO A FUNCTION!!
def get_corr(outcomes, m1, m2):
    if M <= 1: return "Use more than one outcome!"
    else:
        correlation_list = []
        for i in range(len(outcomes)):
            # outcomes for unit i
            outcome_list = outcomes[i]

            # just want m1 and m2
            outcome_0 = torch.tensor(outcome_list[:, m1])
            outcome_1 = torch.tensor(outcome_list[:, m2])

            # calculate the correlation
            mu0 = torch.mean(outcome_0)
            mu1 = torch.mean(outcome_1)
            correlation = torch.dot(outcome_0 - mu0, outcome_1 - mu1) / ((torch.std(outcome_0) * torch.std(outcome_1)))

            correlation_list.append(correlation/(T-1))
    return correlation_list

# check there is correlation between the outcomes!
f1,f2,f3 = generate_factors()
outcomes = generate_noiseless_outcomes(generate_loadings(5),f1,f2,f3)

corr_list = get_corr(outcomes, 2,3)
for i in range(len(corr_list)):
    print(corr_list[i], "\n")

tensor(1.00, dtype=torch.float64) 

tensor(0.60, dtype=torch.float64) 

tensor(0.36, dtype=torch.float64) 

tensor(0.65, dtype=torch.float64) 

tensor(0.98, dtype=torch.float64) 

tensor(0.99, dtype=torch.float64) 

tensor(0.74, dtype=torch.float64) 

tensor(0.98, dtype=torch.float64) 



In [ ]:
#------------------- Run the Model ----------------------------
bias_dict = {}
n_iter = 50

# chain together the data generating functions
def generate_data(m):
    f1,f2,f3 = generate_factors()
    treated_outcomes_list, control_outcomes_list = reshape_data(generate_noiseless_outcomes(generate_loadings(m),f1,f2,f3))
    return treated_outcomes_list, control_outcomes_list

# calculate average bias over increasing numbers of outcomes
for m in 2,10:
    bias_list = []
    for r in range(n_iter): 
        # generate the data
        treated_outcomes_list, control_outcomes_list = generate_data(m)
        # fit the model
        Q_final, w_final = lqOrth.learnQorthogonal(
            # only train on pre-treatment data
            targets = treated_outcomes_list[:T-1],      
            covariates = control_outcomes_list[:T-1],
            embedding_dim = N-1,
            n_iterations = 2000,
            reg_Q = 0,
            reg_w = 0,
            verbose = False,
            init_Q = "eye"
        )

        # calculate the bias for this iteration
        true = treated_outcomes_list[T-1].squeeze()
        pred = (control_outcomes_list[T-1] @ Q_final @ w_final).squeeze()
        rmse_bias = torch.sqrt(torch.mean((true - pred)**2)).item()
        bias_list.append(rmse_bias)

    # calculate mean bias
    mean_bias = sum(bias_list) / len(bias_list)
    bias_dict[m] = mean_bias

print(bias_dict)